In [1]:
# pip install pandas pyarrow xarray netcdf4
import re
import numpy as np
import pandas as pd
import xarray as xr

In [2]:
def df_to_netcdf_ragged(
    parquet_path: str,
    out_nc_path: str,
    time_step_minutes: int = 15,
    start_time_col: str = "track_start_time",
    list_col_hints=None,  # e.g. ["ice_frac_hist", "cot_hist"]
):
    """
    Convert a complex Parquet DataFrame to NetCDF using CF ragged-contiguous arrays
    (one ragged time-series per list-like column). Time for each series starts at
    the row's `start_time_col` and advances in `time_step_minutes`.
    """
    df = pd.read_parquet(parquet_path)
    df.columns = [str(c) for c in df.columns]

    # --- find/normalize the start-time column ---
    if start_time_col not in df.columns:
        # fall back to the first *start*time* looking column if needed
        candidates = [c for c in df.columns if "start" in c.lower() and "time" in c.lower()]
        if candidates:
            start_time_col = candidates[0]
        else:
            raise KeyError(
                f"'{start_time_col}' not found and no obvious start-time column present."
            )
    df[start_time_col] = pd.to_datetime(df[start_time_col], errors="coerce", utc=False)

    # --- helpers to detect & parse list-like cells ----------------------------
    def looks_like_seq_series(series: pd.Series) -> bool:
        if list_col_hints and series.name in list_col_hints:
            return True
        for v in series.dropna().head(10):
            if isinstance(v, (list, tuple, np.ndarray)):
                return True
            if isinstance(v, str) and v.strip().startswith("["):
                return True
        return False

    def parse_seq_cell(cell):
        """Turn a cell into a 1D float array, handling real lists/arrays and stringified lists.
        Accepts comma- or space-separated lists; understands 'nan'."""
        if cell is None or (isinstance(cell, float) and np.isnan(cell)):
            return np.array([], dtype=float)
        if isinstance(cell, (list, tuple, np.ndarray)):
            return np.array(cell, dtype=float).reshape(-1)
        if isinstance(cell, (int, float, np.integer, np.floating)):
            # treat a lone number as a 1-element series
            return np.array([float(cell)], dtype=float)
        if isinstance(cell, str):
            s = cell.strip()
            if s == "" or s.lower() in {"nan", "none"}:
                return np.array([], dtype=float)
            # strip wrapping quotes
            if (s.startswith('"') and s.endswith('"')) or (s.startswith("'") and s.endswith("'")):
                s = s[1:-1].strip()
            # drop surrounding brackets
            inner = s[1:-1] if (s.startswith("[") and s.endswith("]")) else s
            inner = re.sub(r"[\s\t\n\r]+", " ", inner.strip())
            arr = np.fromstring(inner, sep=",")
            if arr.size <= 1:
                arr = np.fromstring(inner, sep=" ")
            if arr.size <= 1 and ";" in inner:
                arr = np.fromstring(inner.replace(";", " "), sep=" ")
            return arr.astype(float, copy=False)
        # fallback
        return np.array([], dtype=float)

    # --- identify which columns are list-like ---------------------------------
    seq_cols = [c for c in df.columns if looks_like_seq_series(df[c])]

    # --- start building the xarray.Dataset ------------------------------------
    nrow = len(df)
    ds = xr.Dataset()
    ds = ds.assign_coords(row=("row", np.arange(nrow, dtype=np.int64)))

    # Keep original index as an ID if it's meaningful
    if df.index.name or not np.array_equal(df.index.values, np.arange(nrow)):
        ds["row_id"] = xr.DataArray(
            df.index.astype(str).to_numpy(), dims=("row",), attrs={"cf_role": "timeseries_id"}
        )

    # Add per-row scalar columns (everything that's NOT list-like)
    for col in df.columns:
        if col in seq_cols:
            continue
        s = df[col]

        # datetimes (already parsed above for start_time_col)
        if np.issubdtype(s.dtype, np.datetime64):
            ds[col] = xr.DataArray(
                s.to_numpy(), dims=("row",), attrs={"standard_name": "time"} if "time" in col.lower() else {}
            )
            continue

        # timedeltas -> seconds
        if np.issubdtype(s.dtype, np.timedelta64):
            ds[col] = xr.DataArray(s.dt.total_seconds().to_numpy(), dims=("row",), attrs={"units": "seconds"})
            continue

        # parse datetimes hiding in object columns named like *time*
        if s.dtype == "O" and "time" in col.lower():
            parsed = pd.to_datetime(s, errors="coerce", utc=False)
            if parsed.notna().sum() > 0:
                ds[col] = xr.DataArray(parsed.to_numpy(), dims=("row",))
                continue

        # booleans -> int8 (NetCDF-friendly)
        if s.dtype == bool or s.dropna().map(lambda x: isinstance(x, (bool, np.bool_))).all():
            ds[col] = xr.DataArray(
                s.fillna(False).astype(np.int8).to_numpy(),
                dims=("row",),
                attrs={"long_name": col, "flag_meanings": "false true"},
            )
            continue

        # numerics
        if np.issubdtype(s.dropna().infer_objects().dtype, np.number):
            ds[col] = xr.DataArray(pd.to_numeric(s, errors="coerce").to_numpy(), dims=("row",))
        else:
            # strings / mixed -> coerce to str
            ds[col] = xr.DataArray(s.astype(str).replace("nan", "").to_numpy(), dims=("row",))

    # Global metadata
    ds.attrs.update(
        {
            "Conventions": "CF-1.8",
            "title": "Converted from Parquet with ragged arrays for per-row time series",
            "featureType": "timeSeries",
            "history": f"Created by df_to_netcdf_ragged; time step = {time_step_minutes} minutes",
        }
    )

    # --- add one ragged time-series per list column ---------------------------
    starts = df[start_time_col]

    for col in seq_cols:
        arrays = [parse_seq_cell(v) for v in df[col].tolist()]
        lengths = np.array([len(a) for a in arrays], dtype=np.int64)
        total = int(lengths.sum())

        # always add the row-size variable (CF contiguous ragged array count)
        ds[f"row_size_{col}"] = xr.DataArray(
            lengths, dims=("row",), attrs={"sample_dimension": f"obs_{col}", "long_name": f"samples for {col} per row"}
        )

        if total == 0:
            continue  # nothing to store for this variable

        flat = np.empty(total, dtype=float)
        times_list = []
        idx = 0

        for st, arr in zip(starts, arrays):
            L = len(arr)
            if L == 0:
                continue
            flat[idx : idx + L] = arr

            # build per-sample time for this row & variable
            if pd.isna(st):
                times_list.append(np.full(L, np.datetime64("NaT", "ns")))
            else:
                base = np.datetime64(st.to_datetime64()) if hasattr(st, "to_datetime64") else np.datetime64(st)
                base_m = base.astype("datetime64[m]")
                # advance in fixed 15-min steps (or whatever you pass)
                t_m = base_m + np.arange(L) * np.timedelta64(time_step_minutes, "m")
                times_list.append(t_m.astype("datetime64[ns]"))

            idx += L

        times_flat = np.concatenate(times_list).astype("datetime64[ns]")

        # create this variable's observation dimension
        dim = f"obs_{col}"
        ds = ds.assign_coords(**{dim: (dim, np.arange(total, dtype=np.int64))})

        # attach time coordinate & the data
        ds[f"time_{col}"] = xr.DataArray(
            times_flat, dims=(dim,), attrs={"standard_name": "time", "long_name": f"time for {col}"}
        )
        ds[col] = xr.DataArray(flat, dims=(dim,), attrs={"coordinates": f"time_{col}"})

    # encode times cleanly; xarray will convert datetime64 to CF time units
    encoding = {v: {"dtype": "int64"} for v in ds.data_vars if str(v).startswith("time_")}

    ds.to_netcdf(out_nc_path, engine="netcdf4", encoding=encoding)
    return ds


# ----------------------- Example -----------------------
# ds = df_to_netcdf_ragged(
#     parquet_path="your_dataframe.parquet",
#     out_nc_path="tracks.nc",
#     time_step_minutes=15,
#     start_time_col="track_start_time",   # change if your column is named differently
#     list_col_hints=None                  # or e.g. ["ice_frac_hist","cot_hist","ctp_hist"]
# )
# print(ds)


In [4]:
df = pd.read_parquet("/cluster/work/climate/dnikolo/Cloud_analysis/np/20070101.0000_20070115.0000/Agg_03_T_06_00.parquet")

In [3]:
df_to_netcdf_ragged("/cluster/work/climate/dnikolo/Cloud_analysis/np/20070101.0000_20070115.0000/Agg_03_T_06_00.parquet",
                    "/cluster/work/climate/dnikolo/dumps/test_out.nc")

/cluster/work/climate/dnikolo/dump/ipykernel_360247/3277581901.py:90: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  ds[col] = xr.DataArray(
/cluster/work/climate/dnikolo/dump/ipykernel_360247/3277581901.py:110: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s.fillna(False).astype(np.int8).to_numpy(),
/cluster/work/climate/dnikolo/dump/ipykernel_360247/3277581901.py:110

<xarray.Dataset> Size: 46MB
Dimensions:                      (row: 14287, obs_start_ice_fraction: 57148,
                                  obs_end_ice_fraction: 57148,
                                  obs_ice_frac_hist: 137815,
                                  obs_cot_hist: 137815,
                                  obs_cot_std_hist: 137815,
                                  obs_cot_nan_frac_hist: 137815,
                                  ...
                                  obs_ctp_std_hist: 137815,
                                  obs_ctp_nan_frac_hist: 137815,
                                  obs_ctt_hist: 137815,
                                  obs_ctt_std_hist: 137815,
                                  obs_lat_hist: 137815, obs_lon_hist: 137815,
                                  obs_size_hist_km: 137815)
Coordinates: (12/15)
  * row                          (row) int64 114kB 0 1 2 3 ... 14284 14285 14286
  * obs_start_ice_fraction       (obs_start_ice_fraction) int64 457kB 0 ... 5...
  * obs_end_ice_fraction         (obs_end_ice_fraction) int64 457kB 0 ... 57147
  * obs_ice_frac_hist            (obs_ice_frac_hist) int64 1MB 0 1 ... 137814
  * obs_cot_hist                 (obs_cot_hist) int64 1MB 0 1 ... 137813 137814
  * obs_cot_std_hist             (obs_cot_std_hist) int64 1MB 0 1 ... 137814
    ...                           ...
  * obs_ctp_nan_frac_hist        (obs_ctp_nan_frac_hist) int64 1MB 0 ... 137814
  * obs_ctt_hist                 (obs_ctt_hist) int64 1MB 0 1 ... 137813 137814
  * obs_ctt_std_hist             (obs_ctt_std_hist) int64 1MB 0 1 ... 137814
  * obs_lat_hist                 (obs_lat_hist) int64 1MB 0 1 ... 137813 137814
  * obs_lon_hist                 (obs_lon_hist) int64 1MB 0 1 ... 137813 137814
  * obs_size_hist_km             (obs_size_hist_km) int64 1MB 0 1 ... 137814
Data variables: (12/66)
    tracknumber                  (row) int64 114kB 1 2 3 4 ... 14285 14286 14287
    is_large_pix_cloud           (row) int8 14kB 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0
    is_cot_valid_cloud           (row) int8 14kB 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0
    is_ctp_valid_cloud           (row) int8 14kB 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0
    is_liq                       (row) int8 14kB 1 1 1 0 0 1 1 ... 1 0 1 1 0 0 0
    is_mix                       (row) int8 14kB 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0
    ...                           ...
    row_size_lon_hist            (row) int64 114kB 352 31 48 52 13 ... 4 4 4 4 4
    time_lon_hist                (obs_lon_hist) datetime64[ns] 1MB 2007-01-01...
    lon_hist                     (obs_lon_hist) float64 1MB -60.29 ... 25.21
    row_size_size_hist_km        (row) int64 114kB 352 31 48 52 13 ... 4 4 4 4 4
    time_size_hist_km            (obs_size_hist_km) datetime64[ns] 1MB 2007-0...
    size_hist_km                 (obs_size_hist_km) float64 1MB 3.707e+05 ......
Attributes:
    Conventions:  CF-1.8
    title:        Converted from Parquet with ragged arrays for per-row time ...
    featureType:  timeSeries
    history:      Created by df_to_netcdf_ragged; time step = 15 minutes

In [2]:
# pip install pandas pyarrow xarray netcdf4
import numpy as np
import pandas as pd
import xarray as xr

def df_to_netcdf_uniform_time(
    parquet_path: str,
    out_nc_path: str,
    time_step_minutes: int = 15,
    start_time_col: str = "track_start_time",
    list_col_hints=None,  # optional: list of columns you know are time series
):
    df = pd.read_parquet(parquet_path)
    df.columns = [str(c) for c in df.columns]
    df[start_time_col] = pd.to_datetime(df[start_time_col], errors="coerce")

    # Detect list-like columns (or use hints)
    def looks_like_seq(series: pd.Series) -> bool:
        if list_col_hints and series.name in list_col_hints:
            return True
        for v in series.dropna().head(10):
            if isinstance(v, (list, tuple, np.ndarray)):
                return True
            if isinstance(v, str) and v.strip().startswith("["):
                return True
        return False

    import re
    def parse_seq(cell):
        if cell is None or (isinstance(cell, float) and np.isnan(cell)):
            return np.array([], dtype=float)
        if isinstance(cell, (list, tuple, np.ndarray)):
            return np.asarray(cell, dtype=float).ravel()
        if isinstance(cell, (int, float, np.integer, np.floating)):
            return np.array([float(cell)], dtype=float)
        if isinstance(cell, str):
            s = cell.strip().strip('"').strip("'")
            inner = s[1:-1] if s.startswith("[") and s.endswith("]") else s
            inner = re.sub(r"\s+", " ", inner)
            arr = np.fromstring(inner, sep=",")
            if arr.size <= 1:
                arr = np.fromstring(inner, sep=" ")
            return arr.astype(float)
        return np.array([], dtype=float)

    seq_cols = [c for c in df.columns if looks_like_seq(df[c])]
    nrow = len(df)

    # Determine per-row number of steps (take the max length across all list columns for that row)
    step = pd.Timedelta(minutes=time_step_minutes)
    row_lengths = np.zeros(nrow, dtype=int)
    parsed_per_col = {}
    for col in seq_cols:
        parsed = [parse_seq(v) for v in df[col].tolist()]
        parsed_per_col[col] = parsed
        row_lengths = np.maximum(row_lengths, np.array([len(a) for a in parsed], dtype=int))

    # Global time axis from earliest start to latest row end (inclusive)
    row_starts = df[start_time_col]
    row_ends = [
        (st + (L - 1) * step) if pd.notna(st) and L > 0 else pd.NaT
        for st, L in zip(row_starts, row_lengths)
    ]
    t0 = pd.to_datetime(row_starts.min())
    t1 = pd.to_datetime(pd.Series(row_ends, dtype="datetime64[ns]").max())
    if pd.isna(t0) or pd.isna(t1):
        raise ValueError("Cannot build a global time axis: missing start or lengths.")
    time = pd.date_range(t0, t1, freq=f"{time_step_minutes}min")

    # Create Dataset with (row, time)
    ds = xr.Dataset()
    ds = ds.assign_coords(row=("row", np.arange(nrow, dtype=np.int64)))
    ds = ds.assign_coords(time=("time", time.to_numpy()))
    ds.attrs.update({"Conventions": "CF-1.8", "featureType": "timeSeries"})

    # Add scalar (per-row) columns
    for col in df.columns:
        if col in seq_cols:
            continue
        s = df[col]
        if np.issubdtype(s.dtype, np.datetime64):
            ds[col] = xr.DataArray(s.to_numpy(), dims=("row",))
        elif np.issubdtype(s.dtype, np.timedelta64):
            ds[col] = xr.DataArray(s.dt.total_seconds().to_numpy(), dims=("row",), attrs={"units": "seconds"})
        elif s.dtype == bool or s.dropna().map(lambda x: isinstance(x, (bool, np.bool_))).all():
            ds[col] = xr.DataArray(s.fillna(False).astype(np.int8).to_numpy(), dims=("row",),
                                   attrs={"flag_meanings": "false true"})
        elif np.issubdtype(pd.to_numeric(s, errors="coerce").dtype, np.number):
            ds[col] = xr.DataArray(pd.to_numeric(s, errors="coerce").to_numpy(), dims=("row",))
        else:
            ds[col] = xr.DataArray(s.astype(str).replace("nan", "").to_numpy(), dims=("row",))

    # Place each list column onto the (row, time) grid, padding with NaN
    for col in seq_cols:
        data = np.full((nrow, time.size), np.nan, dtype=float)

        for i in range(nrow):
            st = row_starts.iat[i]
            arr = parsed_per_col[col][i]
            L = len(arr)
            if pd.isna(st) or L == 0:
                continue
            # position where this row starts on the global time axis
            offset = int((st - t0) / step)
            end = offset + L
            if end > time.size:
                # clip if something runs past t1 (shouldn't if t1 was computed from max end)
                end = time.size
                arr = arr[: end - offset]
            data[i, offset:end] = arr

        ds[col] = xr.DataArray(data.T, dims=("time","row"))

    # Optional: row IDs
    if df.index.name or not np.array_equal(df.index.values, np.arange(nrow)):
        ds["row_id"] = xr.DataArray(df.index.astype(str).to_numpy(), dims=("row",), attrs={"cf_role": "timeseries_id"})

    # Encode time as CF-compliant numeric
    ds.to_netcdf(out_nc_path, engine="netcdf4")
    return ds

# Example:
# ds = df_to_netcdf_uniform_time("your_dataframe.parquet", "tracks_uniform.nc", time_step_minutes=15)
# print(ds)


In [4]:
ds = df_to_netcdf_uniform_time("/cluster/work/climate/dnikolo/Cloud_analysis/np/20070101.0000_20070115.0000/Agg_03_T_06_00.parquet",
                            "/cluster/work/climate/dnikolo/dumps/test_out_less_dim.nc")

/cluster/work/climate/dnikolo/dump/ipykernel_372773/2712445289.py:82: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  ds[col] = xr.DataArray(s.to_numpy(), dims=("row",))
/cluster/work/climate/dnikolo/dump/ipykernel_372773/2712445289.py:86: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ds[col] = xr.DataArray(s.fillna(False).astype(np.int8).to_numpy(), dims=("row",),
/clu

In [10]:
# pip install pandas pyarrow xarray netcdf4
import re
import numpy as np
import pandas as pd
import xarray as xr

def df_to_netcdf_time_from_start_time_first(
    parquet_path: str,
    out_nc_path: str,
    time_step_minutes: int = 15,
    start_time_col: str = "track_start_time",
    track_length_col: str = "track_length",
    list_col_hints=None,
    list_dtype=np.float32
):
    df = pd.read_parquet(parquet_path)
    df.columns = [str(c) for c in df.columns]

    # parse starts / (optional) durations
    if start_time_col not in df.columns:
        raise KeyError(f"Missing start time column '{start_time_col}'")
    df[start_time_col] = pd.to_datetime(df[start_time_col], errors="coerce")

    has_track_len = track_length_col in df.columns
    if has_track_len and not np.issubdtype(df[track_length_col].dtype, np.timedelta64):
        df[track_length_col] = pd.to_timedelta(df[track_length_col], errors="coerce")

    step = pd.Timedelta(minutes=time_step_minutes)

    # detect & parse list-like columns
    def looks_like_seq(s: pd.Series) -> bool:
        if list_col_hints and s.name in list_col_hints: return True
        for v in s.dropna().head(12):
            if isinstance(v, (list, tuple, np.ndarray)): return True
            if isinstance(v, str) and v.strip().startswith("["): return True
        return False

    def parse_seq(cell) -> np.ndarray:
        if cell is None or (isinstance(cell, float) and np.isnan(cell)):
            return np.array([], dtype=list_dtype)
        if isinstance(cell, (list, tuple, np.ndarray)):
            return np.asarray(cell, dtype=list_dtype).ravel()
        if isinstance(cell, (int, float, np.integer, np.floating)):
            return np.asarray([cell], dtype=list_dtype)
        if isinstance(cell, str):
            s = cell.strip()
            if not s or s.lower() in {"nan", "none"}:
                return np.array([], dtype=list_dtype)
            if (s[0] in "\"'" and s[-1] == s[0]): s = s[1:-1].strip()
            if s.startswith("[") and s.endswith("]"): s = s[1:-1]
            s = re.sub(r"\s+", " ", s)
            arr = np.fromstring(s, sep=",")
            if arr.size <= 1: arr = np.fromstring(s, sep=" ")
            if arr.size <= 1 and ";" in s: arr = np.fromstring(s.replace(";", " "), sep=" ")
            return arr.astype(list_dtype, copy=False)
        return np.array([], dtype=list_dtype)

    seq_cols = [c for c in df.columns if looks_like_seq(df[c])]
    nrow = len(df)

    parsed_per_col, len_per_col = {}, {}
    for col in seq_cols:
        parsed = [parse_seq(v) for v in df[col].tolist()]
        parsed_per_col[col] = parsed
        len_per_col[col] = np.array([len(a) for a in parsed], dtype=int)

    # row lengths: max(list lengths) vs declared track_length
    if has_track_len:
        steps = (df[track_length_col] // step).astype("Int64").fillna(0).astype(int)
        expected_len = np.maximum(0, steps + (steps > 0).astype(int))
    else:
        expected_len = np.zeros(nrow, dtype=int)

    max_list_len = np.max(np.vstack([len_per_col[c] for c in seq_cols]), axis=0) if seq_cols else np.zeros(nrow, int)
    row_len = np.maximum(expected_len, max_list_len)
    max_len = int(row_len.max()) if nrow else 0

    # --- coordinates: time-first using an integer index ---
    # time_from_start is a pure index (0..max_len-1), avoids timedelta encoding issues
    t_index = np.arange(max_len, dtype=np.int32)  # (time_from_start,)
    ds = xr.Dataset()
    ds = ds.assign_coords(time_from_start=("time_from_start", t_index))
    ds = ds.assign_coords(row=("row", np.arange(nrow, dtype=np.int64)))

    # auxiliary coordinate: elapsed minutes for readability
    if max_len > 0:
        elapsed_minutes = t_index.astype(np.int64) * time_step_minutes
        ds["elapsed_minutes"] = xr.DataArray(elapsed_minutes, dims=("time_from_start",))
        ds["elapsed_minutes"].attrs.update({
            "long_name": "elapsed minutes from track start",
            "units": "minutes"
        })

    # absolute time (time_from_start, row)
    starts = df[start_time_col].to_numpy(dtype="datetime64[ns]")
    if max_len > 0:
        offs_td = (t_index.astype("int64") * time_step_minutes).astype("timedelta64[m]").astype("timedelta64[ns]")
        abs_time = offs_td[:, None] + starts[None, :]  # (time, row)
        ds["time"] = xr.DataArray(abs_time, dims=("time_from_start", "row"))
        ds["time"].attrs.update({"long_name": "absolute time", "standard_name": "time"})

    ds.attrs.update({
        "Conventions": "CF-1.8",
        "featureType": "timeSeries",
        "title": "Tracks on a uniform time-from-start grid (time-first, integer index)",
        "history": f"time_from_start length = {max_len} at {time_step_minutes} min cadence",
    })

    # per-row scalars stay 1-D on 'row'
    for col in df.columns:
        if col in seq_cols: continue
        s = df[col]
        if np.issubdtype(s.dtype, np.datetime64):
            ds[col] = xr.DataArray(s.to_numpy(), dims=("row",))
        elif np.issubdtype(s.dtype, np.timedelta64):
            ds[col] = xr.DataArray(s.dt.total_seconds().to_numpy(), dims=("row",), attrs={"units": "seconds"})
        elif s.dtype == bool or s.dropna().map(lambda x: isinstance(x, (bool, np.bool_))).all():
            ds[col] = xr.DataArray(s.fillna(False).astype(np.int8).to_numpy(), dims=("row",),
                                   attrs={"flag_meanings": "false true"})
        else:
            co = pd.to_numeric(s, errors="coerce")
            if np.issubdtype(co.dtype, np.number):
                ds[col] = xr.DataArray(co.to_numpy(), dims=("row",))
            else:
                ds[col] = xr.DataArray(s.astype(str).replace("nan", "").to_numpy(), dims=("row",))

    # list-like variables as (time_from_start, row)
    for col in seq_cols:
        data = np.full((max_len, nrow), np.nan, dtype=list_dtype)
        Lvec = len_per_col[col]
        for i in range(nrow):
            L = Lvec[i]
            if L <= 0: continue
            data[:min(L, max_len), i] = parsed_per_col[col][i][:max_len]
        ds[col] = xr.DataArray(data, dims=("time_from_start", "row"))
        ds[f"valid_length_{col}"] = xr.DataArray(Lvec, dims=("row",),
                                                attrs={"long_name": f"valid samples in {col} per row"})

    ds = ds.transpose("time_from_start", "row", ...)
    ds.to_netcdf(out_nc_path, engine="netcdf4")
    return ds


In [11]:
ds = df_to_netcdf_time_from_start_time_first("/cluster/work/climate/dnikolo/Cloud_analysis/np/20070101.0000_20070115.0000/Agg_03_T_06_00.parquet",
                            "/cluster/work/climate/dnikolo/dumps/test_out_less_dim_2.nc")

/cluster/work/climate/dnikolo/dump/ipykernel_372773/2273356895.py:114: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  ds[col] = xr.DataArray(s.to_numpy(), dims=("row",))
/cluster/work/climate/dnikolo/dump/ipykernel_372773/2273356895.py:118: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ds[col] = xr.DataArray(s.fillna(False).astype(np.int8).to_numpy(), dims=("row",),
/c

In [12]:
ds

<xarray.Dataset> Size: 325MB
Dimensions:                          (time_from_start: 352, row: 14287)
Coordinates:
  * time_from_start                  (time_from_start) int32 1kB 0 1 ... 350 351
  * row                              (row) int64 114kB 0 1 2 ... 14285 14286
Data variables: (12/54)
    elapsed_minutes                  (time_from_start) int64 3kB 0 15 ... 5265
    time                             (time_from_start, row) datetime64[ns] 40MB ...
    tracknumber                      (row) int64 114kB 1 2 3 ... 14286 14287
    is_large_pix_cloud               (row) int8 14kB 0 0 0 0 0 0 ... 0 0 0 0 0 0
    is_cot_valid_cloud               (row) int8 14kB 0 0 0 0 0 0 ... 0 0 0 0 0 0
    is_ctp_valid_cloud               (row) int8 14kB 0 0 0 0 0 0 ... 0 0 0 0 0 0
    ...                               ...
    lat_hist                         (time_from_start, row) float32 20MB 37.2...
    valid_length_lat_hist            (row) int64 114kB 352 31 48 52 ... 4 4 4 4
    lon_hist                         (time_from_start, row) float32 20MB -60....
    valid_length_lon_hist            (row) int64 114kB 352 31 48 52 ... 4 4 4 4
    size_hist_km                     (time_from_start, row) float32 20MB 3.70...
    valid_length_size_hist_km        (row) int64 114kB 352 31 48 52 ... 4 4 4 4
Attributes:
    Conventions:  CF-1.8
    featureType:  timeSeries
    title:        Tracks on a uniform time-from-start grid (time-first, integ...
    history:      time_from_start length = 352 at 15 min cadence